# 03 — Metrics

NLPD, CRPS, SACC and MMD at K=1 and K=4 for the three methods trained in notebook 02. The metric
code is copied from `compute_metrics.py`.

## Setup and definitions (as in notebooks 01/02)


In [1]:
import math
import itertools
import torch
import torch.nn as nn
import numpy as np
from functools import partial
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

torch.manual_seed(0)
np.random.seed(0)

CKPT_PRETRAINED = Path("../checkerboard_example/checkpoints")
CKPT_OURS = Path("ckpts")

sigma = 0.1
B = 10_000   # observations
J = 100      # posterior samples per observation


def sample_checkerboard(n_samples, n_squares=4):
    x = np.random.rand(n_samples, 2)
    idx = (x * n_squares).astype(int)
    mask = (idx[:, 0] + idx[:, 1]) % 2 == 0
    while not mask.all():
        n_bad = (~mask).sum()
        x_new = np.random.rand(n_bad, 2)
        idx_new = (x_new * n_squares).astype(int)
        x[~mask] = x_new; idx[~mask] = idx_new
        mask = (idx[:, 0] + idx[:, 1]) % 2 == 0
    return 4.0 * (x - np.array([[0.5, 0.5]]))


def sample_y(x, sigma_obs):
    return x[:, :1] + sigma_obs * torch.randn(x.shape[0], 1, device=x.device, dtype=x.dtype)


class MeanFlowMLP(nn.Module):
    def __init__(self, hidden=256, depth=4):
        super().__init__()
        layers = []; dim = 4
        for _ in range(depth):
            layers += [nn.Linear(dim, hidden), nn.SiLU()]; dim = hidden
        layers += [nn.Linear(dim, 2)]
        self.net = nn.Sequential(*layers)
    def forward(self, z, r, t):
        return self.net(torch.cat([z, t, r], dim=1))


class Adapter(nn.Module):
    def __init__(self, hidden=128, depth=3, z_dim=2):
        super().__init__()
        layers = []; dim = 1
        for _ in range(depth):
            layers += [nn.Linear(dim, hidden), nn.SiLU()]; dim = hidden
        self.backbone = nn.Sequential(*layers)
        self.mu = nn.Linear(dim, z_dim)
        self.log_std = nn.Linear(dim, z_dim)
    def forward(self, y):
        h = self.backbone(y)
        return self.mu(h), self.log_std(h).clamp(-6.0, 3.0)


def reparam_sample(mu, log_std, J=None):
    if J is None:
        return mu + torch.exp(log_std) * torch.randn_like(mu)
    B_, D = mu.shape
    eps = torch.randn(B_, J, D, device=mu.device, dtype=mu.dtype)
    return mu.unsqueeze(1) + torch.exp(log_std).unsqueeze(1) * eps


def f_theta(z, u_model, K=1):
    if z.ndim == 3:
        B_, J_, D = z.shape
        z = z.reshape(B_ * J_, D)
    ts = torch.linspace(0.0, 1.0, K + 1, device=z.device)
    for i in range(K, 0, -1):
        t_i = torch.full((z.shape[0], 1), ts[i].item(), device=z.device)
        r_i = torch.full((z.shape[0], 1), ts[i-1].item(), device=z.device)
        z = z - (t_i - r_i) * u_model(z, r_i, t_i)
    if 'J_' in locals():
        z = z.reshape(B_, J_, D)
    return z


def unscale_to_unit(x_scaled):
    return x_scaled / 4.0 + 0.5


device: cuda


## How each metric is computed

Draw 10,000 truth points from the checkerboard and one measurement $y$ for each. For every $y$
the model draws J=100 samples. Each metric gives one scalar per measurement, and the mean over
measurements is reported.

**NLPD.** Draw a second measurement $y'$ of the same truth point, evaluate
$\mathcal{N}(y'; x^{(j)}_1, \sigma^2)$ at each sample, and take minus the log of the mean.

**CRPS.** The mean distance from the samples to the truth point, minus half the mean distance
between pairs of samples.

**SACC.** The fraction of samples landing in a filled cell. The prior version uses unconditional
samples and no measurement.

**MMD.** Unbiased estimator with an RBF kernel and a median-heuristic bandwidth, against true
posterior samples drawn by rejection. Computed for the first 1,000 measurements. The prior
version compares unconditional samples against a fixed truth set.

In [2]:
def nlpd(y, x_samples, sigma_obs):
    B_, J_, D = x_samples.shape
    Ax = x_samples[..., 0]
    y0 = y.squeeze(-1).unsqueeze(1)
    log_norm = -0.5 * math.log(2 * math.pi * (sigma_obs ** 2))
    log_probs = log_norm - 0.5 * (y0 - Ax) ** 2 / (sigma_obs ** 2)
    return -(torch.logsumexp(log_probs, dim=1) - math.log(J_))


def crps(x_true, x_samples):
    B_, J_, D = x_samples.shape
    term1 = torch.mean(torch.norm(x_samples - x_true.unsqueeze(1).expand(B_, J_, D), dim=2), dim=1)
    pairwise = torch.norm(x_samples.unsqueeze(2) - x_samples.unsqueeze(1), dim=3)
    return term1 - 0.5 * torch.mean(pairwise, dim=(1, 2))


def support_accuracy(x01, n_squares=4, filled_parity=0, eps=0.0):
    x = x01
    inside = ((x[..., 0] >= eps) & (x[..., 0] < 1.0 - eps) &
              (x[..., 1] >= eps) & (x[..., 1] < 1.0 - eps))
    ij = torch.floor(x * n_squares).long().clamp(0, n_squares - 1)
    filled = ((ij[..., 0] + ij[..., 1]) % 2 == filled_parity)
    return (inside & filled).float().mean(dim=1)


@torch.no_grad()
def pairwise_sq_dists(x, y):
    x2 = (x**2).sum(dim=1, keepdim=True)
    y2 = (y**2).sum(dim=1, keepdim=True).T
    return x2 + y2 - 2.0 * (x @ y.T)


@torch.no_grad()
def median_heuristic_sigma(x, y, max_samples=2000, eps=1e-12):
    d2 = pairwise_sq_dists(x[:max_samples], y[:max_samples])
    return torch.sqrt(torch.clamp(torch.median(d2), min=eps))


@torch.no_grad()
def mmd_squared(x, y, sigma=None, max_sigma_samples=2000):
    N, M = x.shape[0], y.shape[0]
    if sigma is None:
        sigma = median_heuristic_sigma(x, y, max_samples=max_sigma_samples)
    gamma = 1.0 / (2.0 * sigma**2)
    Kxx = torch.exp(-gamma * pairwise_sq_dists(x, x))
    Kyy = torch.exp(-gamma * pairwise_sq_dists(y, y))
    Kxy = torch.exp(-gamma * pairwise_sq_dists(x, y))
    Kxx = Kxx - torch.diag(torch.diag(Kxx))
    Kyy = Kyy - torch.diag(torch.diag(Kyy))
    return Kxx.sum() / (N * (N - 1)) + Kyy.sum() / (M * (M - 1)) - 2.0 * Kxy.mean()


@torch.no_grad()
def sample_posterior_rejection(y, sigma_obs, J, prior_sampler, proposal_K=4096,
                               max_rounds=10_000, device=None):
    if device is None: device = y.device
    if y.ndim == 1: y = y[:, None]
    y = y.to(device)
    B_ = y.shape[0]
    out = torch.empty(B_, J, 2, device=device)
    filled = torch.zeros(B_, dtype=torch.long, device=device)
    sigma_t = torch.tensor(sigma_obs, device=device, dtype=y.dtype)
    for _ in range(max_rounds):
        if bool((filled >= J).all()): break
        x_prop = prior_sampler(B_ * proposal_K).reshape(B_, proposal_K, 2)
        x_prop = torch.tensor(x_prop, device=device, dtype=torch.float32)
        diff = (y[:, 0:1] - x_prop[:, :, 0]) / sigma_t
        accept = torch.rand(B_, proposal_K, device=device) < torch.exp(-0.5 * diff**2)
        for b in range(B_):
            if filled[b] >= J: continue
            xb_acc = x_prop[b][accept[b]]
            if xb_acc.numel() == 0: continue
            take = min(J - int(filled[b].item()), xb_acc.shape[0])
            out[b, filled[b]:filled[b]+take] = xb_acc[:take]
            filled[b] += take
    assert bool((filled >= J).all())
    return out


## Shared evaluation data

One draw, reused for every method and K, so table rows are directly comparable.


In [3]:
prior_sampler = partial(sample_checkerboard, n_squares=4)
x_prior = torch.tensor(prior_sampler(B), dtype=torch.float32).to(device)
y_val = sample_y(x_prior, sigma)
y_prime = sample_y(x_prior, sigma)

print("Rejection-sampling the true posterior for 1000 observations...")
x_posterior = sample_posterior_rejection(y_val[:1000], sigma, J, prior_sampler)
print("done:", tuple(x_posterior.shape))


Rejection-sampling the true posterior for 1000 observations...


done: (1000, 100, 2)


## Evaluation function

One call evaluates one method at one K, giving one row of the results table.

In [4]:
@torch.no_grad()
def evaluate(model, adapter, K):
    model.eval(); adapter.eval()
    mu_val, log_std_val = adapter(y_val)
    z_val = reparam_sample(mu_val, log_std_val, J=J)   # (B,J,2)
    eps = torch.randn_like(z_val)
    xhat_val = f_theta(z_val, model, K)                # posterior samples
    xhat_uncond = f_theta(eps, model, K)               # prior samples

    res = {}
    res['nlpd'] = float(nlpd(y_prime, xhat_val, sigma).mean())
    res['crps'] = float(crps(x_prior, xhat_val).mean())
    res['sacc_post'] = float(support_accuracy(unscale_to_unit(xhat_val)).mean())
    res['sacc_prior'] = float(support_accuracy(unscale_to_unit(xhat_uncond.transpose(0, 1))).mean())

    post_mmds = [float(torch.sqrt(torch.clamp(mmd_squared(x_posterior[i], xhat_val[i]), min=0)))
                 for i in range(1000)]
    res['mmd_post'] = float(np.mean(post_mmds))

    prior_mmds = []
    for _ in range(100):
        e = torch.randn((B, 2), device=device, dtype=torch.float32)
        prior_mmds.append(float(torch.sqrt(torch.clamp(mmd_squared(x_prior, f_theta(e, model, K)), min=0))))
    res['mmd_prior'] = float(np.mean(prior_mmds))
    return res


## Evaluate all three methods at K=1 and K=4

frozen-θ uses the pretrained `mf_model.pt` with the adapter from `ckpts/adapter_only_model.pt`.
unconstrained-θ uses both networks from `ckpts/joint_adapter_mf_model_naive.pt`. VFM uses both
from `ckpts/vfm_adapter_tau_100.0_alpha_1.0_ema.pt`.

In [5]:
def load_pair(generator_source, adapter_source):
    model = MeanFlowMLP(hidden=512, depth=6).to(device)
    model.load_state_dict(generator_source)
    adapter = Adapter(hidden=256, depth=4, z_dim=2).to(device)
    adapter.load_state_dict(adapter_source)
    return model, adapter

mf_sd = torch.load(CKPT_PRETRAINED / "mf_model.pt", map_location=device)
frozen_ck = torch.load(CKPT_OURS / "adapter_only_model.pt", map_location=device)
naive_ck = torch.load(CKPT_OURS / "joint_adapter_mf_model_naive.pt", map_location=device)
vfm_ck = torch.load(CKPT_OURS / "vfm_adapter_tau_100.0_alpha_1.0_ema.pt", map_location=device)

methods = {
    "frozen-theta": load_pair(mf_sd, frozen_ck["adapter"]),
    "unconstrained-theta": load_pair(naive_ck["mf_model"], naive_ck["adapter"]),
    "VFM": load_pair(vfm_ck["mf_model"], vfm_ck["adapter"]),
}

results = {}
for name, (model, adapter) in methods.items():
    for K in (1, 4):
        print(f"evaluating {name}, K={K} ...")
        results[(name, K)] = evaluate(model, adapter, K)

cols = ['nlpd', 'crps', 'sacc_post', 'sacc_prior', 'mmd_post', 'mmd_prior']
header = f"{'method':22s} {'K':>2s} " + " ".join(f"{c:>10s}" for c in cols)
print(); print(header); print("-" * len(header))
for (name, K), res in results.items():
    print(f"{name:22s} {K:2d} " + " ".join(f"{res[c]:10.4f}" for c in cols))


evaluating frozen-theta, K=1 ...


evaluating frozen-theta, K=4 ...


evaluating unconstrained-theta, K=1 ...


evaluating unconstrained-theta, K=4 ...


evaluating VFM, K=1 ...


evaluating VFM, K=4 ...



method                  K       nlpd       crps  sacc_post sacc_prior   mmd_post  mmd_prior
-------------------------------------------------------------------------------------------
frozen-theta            1    -0.4610     0.7607     0.8965     0.8690     0.3036     0.0178
frozen-theta            4    -0.4345     0.7575     0.9274     0.8933     0.2990     0.0119
unconstrained-theta     1    -0.5423     0.6994     0.4722     0.4696     0.2157     0.0649
unconstrained-theta     4     0.2253     0.7014     0.5432     0.5412     0.2086     0.0721
VFM                     1    -0.5390     0.6168     0.8867     0.8846     0.0377     0.0122
VFM                     4    -0.5457     0.6143     0.9153     0.9040     0.0307     0.0053
